# **Diffie-Hellman, RSA et ECDSA**

Implémentation from scratch (Python standard uniquement) de l'échange de clés Diffie-Hellman, du chiffrement RSA et de la signature ECDSA.

> Les paramètres (`N_ETUDIANT`, date de naissance) sont **fictifs**. Remplacez-les dans la cellule « Paramètres utilisateur » pour obtenir vos propres résultats.
> Les tailles de clés sont volontairement minuscules : code pédagogique, à ne pas utiliser en production.


In [1]:

import random
import hashlib
from math import isqrt


## PARTIE 1: Diffie-Hellman



In [2]:
# =============================
# PARAMÈTRES UTILISATEUR
# =============================
# - N_ETUDIANT : servira à chercher des nombres premiers proches
# - JOUR/MOIS/ANNEE : utiles pour définir x=JJMM, y=AAAA (DH) ou M=JJMMYYYY (RSA)
N_ETUDIANT = 12345678    # ← remplace par ton numéro étudiant N (entier)
JOUR_NAISS   = 15          # ← JJ
MOIS_NAISS   = 7           # ← MM
ANNEE_NAISS  = 2001        # ← AAAA

# Option d’égalité : si deux premiers sont à égale distance de N,
# choisir le plus grand ? (True => plus grand, False => plus petit)
# Permet de lever l'ambiguïté quand N est pile au milieu de deux premiers.
TIE_BREAK_TO_HIGHER = True


# =============================
# OUTILS NOMBRES PREMIERS
# =============================
def _miller_rabin_witness(a, d, n, s):
    """Sous-routine Miller-Rabin."""
    # Teste si 'a' est un témoin de composité pour n.
    # Hypothèse : n-1 = 2^s * d (avec d impair). On vérifie si a^d ≡ 1 (mod n)
    # ou si a^(2^r * d) ≡ -1 (mod n) pour un certain r < s. Sinon, n est composé.
    x = pow(a, d, n)
    if x == 1 or x == n - 1:
        return False
    for _ in range(s - 1):
        x = (x * x) % n
        if x == n - 1:
            return False
    return True

def is_probable_prime(n: int) -> bool:
    """Test de primalité probabiliste (déterministe pour n < 2^64)."""
    # Étapes :
    # 1) Filtres rapides par petits diviseurs (accélère énormément).
    # 2) Décomposition n-1 = 2^s * d.
    # 3) Miller-Rabin avec bases "suffisantes" → déterministe sous 2^64.
    if n < 2:
        return False
    small_primes = [2,3,5,7,11,13,17,19,23,29,31]
    for p in small_primes:
        if n % p == 0:
            return n == p
    # décomposition n-1 = 2^s * d
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    # Bases déterministes suffisantes pour 64 bits
    for a in [2, 3, 5, 7, 11, 13, 17]:
        if a % n == 0:
            continue
        if _miller_rabin_witness(a, d, n, s):
            return False
    return True

def next_prime_ge(n: int) -> int:
    """Plus petit premier ≥ n."""
    # Stratégie : partir de n (ou n+1 si pair) puis tester seulement les impairs.
    if n <= 2:
        return 2
    p = n if n % 2 else n + 1
    while not is_probable_prime(p):
        p += 2
    return p

def prev_prime_le(n: int) -> int:
    """Plus grand premier ≤ n."""
    # Stratégie symétrique : descendre par pas de 2 (impairs) jusqu'à trouver un premier.
    if n < 2:
        raise ValueError("Il n'existe pas de premier ≤ 1.")
    p = n if n % 2 else n - 1
    while p >= 2 and not is_probable_prime(p):
        p -= 2
    if p < 2:
        raise ValueError("Aucun premier trouvé ≤ n.")
    return p

def nearest_prime(n: int, tie_break_to_higher: bool = True) -> int:
    """
    Premier p le plus proche de n.
    En cas d'égalité des distances, choisir le plus grand si tie_break_to_higher=True,
    sinon le plus petit.
    """
    # Idée : on calcule le premier au-dessous (lo) et au-dessus (hi), puis on compare |n-lo| et |hi-n|.
    # Cas n<2 : par convention, renvoie 2 (premier minimal).
    if n < 2:
        return 2
    lo = prev_prime_le(n)
    hi = next_prime_ge(n)
    dlo = abs(n - lo)
    dhi = abs(hi - n)
    if dlo < dhi:
        return lo
    if dhi < dlo:
        return hi
    # égalité
    return hi if tie_break_to_higher else lo


# =============================
# OUTILS GROUPES & GÉNÉRATEURS
# =============================
def factorize(n: int):
    """Factorisation naïve (suffisante pour p-1 de taille modérée). Renvoie l'ensemble des facteurs premiers."""
    # But : obtenir les facteurs premiers de φ=p-1 pour tester les racines primitives.
    # Méthode simple (trial division) : OK car p-1 reste de taille raisonnable dans l'exo.
    factors = set()
    d = 2
    m = n
    while d * d <= m:
        while m % d == 0:
            factors.add(d)
            m //= d
        d += 1 if d == 2 else 2  # après 2, tester seulement les impairs
    if m > 1:
        factors.add(m)
    return factors

def primitive_root_mod_prime(p: int) -> int:
    """
    Trouve un générateur g du groupe multiplicatif (Z/pZ)*.
    g est générateur si g^((p-1)/q) != 1 (mod p) pour tout q | (p-1).
    """
    # Rappel : (Z/pZ)* est cyclique de taille φ=p-1. Un g est générateur si
    # pour chaque facteur premier q de φ, g^{φ/q} ≠ 1 (mod p).
    if p == 2:
        return 1
    phi = p - 1
    prime_factors = factorize(phi)
    g = 2
    while g < p:
        ok = True
        for q in prime_factors:
            if pow(g, phi // q, p) == 1:
                ok = False
                break
        if ok:
            return g
        g += 1
    raise RuntimeError("Aucun générateur trouvé (improbable pour un p premier).")


### QUESTION 1: Trouver le nombre premier p le plus proche de votre numéro étudiant N

In [3]:

# - tie_break_to_higher : si égalité de distance entre 2 premiers, on prend le plus grand si True.

p = nearest_prime(N_ETUDIANT, tie_break_to_higher=TIE_BREAK_TO_HIGHER)

# --- Affichage pédagogique des résultats de Q1 ---
# On rappelle la valeur de N, puis on montre p trouvé + la règle de départage (plus grand / plus petit).

print("===== Q1 : Premier p le plus proche de N =====")
print(f"N = {N_ETUDIANT}")
print(f"p = {p} (premier le plus proche de N, tie→ {'plus grand' if TIE_BREAK_TO_HIGHER else 'plus petit'})")


===== Q1 : Premier p le plus proche de N =====
N = 12345678
p = 12345701 (premier le plus proche de N, tie→ plus grand)


### QUESTION 2 : Déterminer un générateur g du groupe Z/pZ

In [4]:

g = primitive_root_mod_prime(p)

print("\n===== Q2 : Générateur g de (Z/pZ)* =====")
print(f"p = {p}, p-1 = {p-1}")
print(f"g = {g} (vérifié générateur)")




===== Q2 : Générateur g de (Z/pZ)* =====
p = 12345701, p-1 = 12345700
g = 2 (vérifié générateur)


### QUESTION 3: Alice et Bob ont comme clés secrètes x= votre jour et mois de naissance , y = votre année de naissance. Calculer les clés publique de Alice et Bob

In [5]:

# Convention pour x : concaténation JJMM (ex: 15/07 -> x = 1507).
# Convention pour y : AAAA (Mon année de naissance).
x = 100 * JOUR_NAISS + MOIS_NAISS   # JJMM
y = ANNEE_NAISS                      # AAAA

A_public = pow(g, x, p)  # clé publique d’Alice
B_public = pow(g, y, p)  # clé publique de Bob

print("\n===== Q3 : Clés publiques =====")
print(f"Date de naissance = {JOUR_NAISS:02d}/{MOIS_NAISS:02d}/{ANNEE_NAISS}")
print(f"x = JJMM = {x}, y = AAAA = {y}")
print(f"Clé publique d’Alice (A = g^x mod p) = {A_public}")
print(f"Clé publique de Bob   (B = g^y mod p) = {B_public}")




===== Q3 : Clés publiques =====
Date de naissance = 15/07/2001
x = JJMM = 1507, y = AAAA = 2001
Clé publique d’Alice (A = g^x mod p) = 3829910
Clé publique de Bob   (B = g^y mod p) = 5232420


### QUESTION 4: Calculer la clé secrète commune d'Alice et Bob

In [6]:

# Alice calcule s_A = B^x mod p ; Bob calcule s_B = A^y mod p.
# Par Diffie–Hellman, s_A = s_B.
s_A = pow(B_public, x, p)
s_B = pow(A_public, y, p)

print("\n===== Q4 : Clé secrète commune =====")
print(f"s_A = B^x mod p = {s_A}")
print(f"s_B = A^y mod p = {s_B}")
print("Vérification :", "OK (égales)" if s_A == s_B else " Problème — elles devraient être égales")




===== Q4 : Clé secrète commune =====
s_A = B^x mod p = 10303966
s_B = A^y mod p = 10303966
Vérification : OK (égales)


## PARTIE 2 : RSA

In [7]:
# =============================
# OUTILS : arithmétique modulaire (PGCD, EEA, exp rapide)
# =============================

def gcd(a: int, b: int) -> int:
    #  PGCD classique (algorithme d'Euclide)
    # Boucle : on remplace (a,b) par (b, a mod b) jusqu'à ce que b=0.
    # À la fin, a = gcd(a,b). On renvoie la valeur absolue.
    while b:
        a, b = b, a % b
    return abs(a)

def egcd(a: int, b: int):
    """Algorithme d'Euclide étendu. Renvoie (g, x, y) tel que ax + by = g = gcd(a,b)."""
    # 👉 Version étendue : en plus du PGCD g, on calcule x et y qui vérifient ax + by = g.
    # Utile pour trouver des inverses modulo (quand g=1).
    old_r, r = a, b     # rémainders successifs
    old_s, s = 1, 0     # coefficients pour 'a'
    old_t, t = 0, 1     # coefficients pour 'b'
    while r != 0:
        q = old_r // r                  # quotient de division euclidienne
        old_r, r = r, old_r - q * r     # mise à jour des restes
        old_s, s = s, old_s - q * s     # mise à jour des coeff. pour 'a'
        old_t, t = t, old_t - q * t     # mise à jour des coeff. pour 'b'
    return old_r, old_s, old_t  # g, x, y (à la fin : old_r=gcd, old_s=x, old_t=y)

def modinv(a: int, m: int) -> int:
    #  Inverse modulaire : retourne a^{-1} mod m si gcd(a,m)=1
    # On utilise egcd pour récupérer x tel que ax + my = 1 ⇒ x ≡ a^{-1} (mod m)
    g, x, _ = egcd(a, m)
    if g != 1:
        # Si le PGCD n'est pas 1, l'inverse n'existe pas (a et m ne sont pas copremiers).
        raise ValueError(f"Pas d'inverse mod {m} pour a={a} (gcd={g}).")
    return x % m  # normalisation dans [0, m-1]

def fast_pow_mod(base: int, exp: int, mod: int) -> int:
    """Exponentiation rapide binaire (pour montrer la méthode)."""
    #  Calcul de base^exp (mod mod) en O(log exp)
    # Technique "square-and-multiply" : on parcourt les bits de exp.
    result = 1
    b = base % mod  # réduction initiale
    e = exp
    while e > 0:
        if e & 1:                 # si le bit courant de e est 1 → on multiplie dans l'accumulateur
            result = (result * b) % mod
        b = (b * b) % mod         # square : on met à jour b ← b^2 mod mod
        e >>= 1                   # shift à droite : on passe au bit suivant
    return result                 # résultat final = base^exp mod mod


### QUESTION 1: Trouver les nombre premiers p  et q les plus proches de N/2 et 2N respectivement.


In [8]:
# --- Cibles pour RSA à partir de N ---
# Idée : on veut p ≈ N/2 et q ≈ 2N, puis on prendra le "premier le plus proche" de ces cibles.
target_p_rsa = max(2, N_ETUDIANT // 2)
target_q_rsa = max(3, 2 * N_ETUDIANT)

# --- Trouver les premiers p_rsa et q_rsa les plus proches des cibles ---
p_rsa = nearest_prime(target_p_rsa, tie_break_to_higher=TIE_BREAK_TO_HIGHER)
q_rsa = nearest_prime(target_q_rsa, tie_break_to_higher=TIE_BREAK_TO_HIGHER)

# --- Sécurité de principe : éviter p_rsa == q_rsa (cas quasi impossible ici) ---
# Si cela arrive, on décale légèrement la cible de q_rsa et on reprend la recherche.
if p_rsa == q_rsa:
    # décale la cible de q_rsa
    q_rsa = nearest_prime(target_q_rsa + 1, tie_break_to_higher=TIE_BREAK_TO_HIGHER)

# --- Calculs RSA de base : n et phi(n) ---
n_rsa = p_rsa * q_rsa
phi_rsa = (p_rsa - 1) * (q_rsa - 1)

# --- Affichage pédagogique des résultats ---
print("===== Q1 : Choix de p, q =====")
print(f"N = {N_ETUDIANT}")
print(f"Cible p ≈ N/2 → {target_p_rsa}  → p_rsa = {p_rsa} (premier le plus proche)")
print(f"Cible q ≈ 2N  → {target_q_rsa}  → q_rsa = {q_rsa} (premier le plus proche)")
print(f"Modulus n_rsa = p_rsa*q_rsa = {n_rsa}")
print(f"φ(n_rsa) = (p_rsa-1)(q_rsa-1) = {phi_rsa}")


===== Q1 : Choix de p, q =====
N = 12345678
Cible p ≈ N/2 → 6172839  → p_rsa = 6172847 (premier le plus proche)
Cible q ≈ 2N  → 24691356  → q_rsa = 24691357 (premier le plus proche)
Modulus n_rsa = p_rsa*q_rsa = 152415968983379
φ(n_rsa) = (p_rsa-1)(q_rsa-1) = 152415938119176


### QUESTION 2 : Determiner l'entier le plus petit e premier avec (p-1)(q-1)

In [9]:
# φ(n_rsa) est pair (p_rsa et q_rsa impairs) donc e_rsa=2 est exclu : on part de 3
# et on incrémente jusqu'au premier entier copremier avec φ(n_rsa).
e_rsa = 3
while gcd(e_rsa, phi_rsa) != 1:
    e_rsa += 1

print("\n===== Q2 : Exposant public e =====")
print(f"e_rsa choisi (plus petit copremier avec φ(n_rsa)) = {e_rsa}")
print(f"vérif: gcd(e_rsa, φ(n_rsa)) = {gcd(e_rsa, phi_rsa)}")


===== Q2 : Exposant public e =====
e_rsa choisi (plus petit copremier avec φ(n_rsa)) = 5
vérif: gcd(e_rsa, φ(n_rsa)) = 1


### QUESTION 3: Calculer par l'algorithme d'Euclide étendu d tel que ed ≡ 1 mod φ(n)

In [10]:
# Euclide étendu : e_rsa*d_rsa ≡ 1 (mod phi_rsa)
d_rsa = modinv(e_rsa, phi_rsa)

print("\n===== Q3 : Exposant privé d =====")
print(f"d_rsa = e_rsa^(-1) mod phi_rsa = {d_rsa}")
print(f"vérif: (e_rsa*d_rsa) % phi_rsa = {(e_rsa*d_rsa) % phi_rsa} (doit valoir 1)")


===== Q3 : Exposant privé d =====
d_rsa = e_rsa^(-1) mod phi_rsa = 121932750495341
vérif: (e_rsa*d_rsa) % phi_rsa = 1 (doit valoir 1)


### QUESTION 4: Chiffrer avec la clé publique de naissance  M puis déchiffrer le cryptogramme par l'algorithme  d'exponentiation rapide

In [11]:
# Message M = JJMMYYYY (date de naissance)
# Si M >= n_rsa, on travaille avec M_mod = M % n_rsa (principe de découpage en blocs < n_rsa)
# ================================================================
M = 10_000 * (100 * JOUR_NAISS + MOIS_NAISS) + ANNEE_NAISS  # JJMMYYYY
M_mod = M % n_rsa

C = fast_pow_mod(M_mod, e_rsa, n_rsa)   # chiffrement (on aurait pu utiliser pow(M_mod, e, n))
M_dec = fast_pow_mod(C, d_rsa, n_rsa)   # déchiffrement

print("\n===== Q4 : Chiffrement / Déchiffrement =====")
print(f"Date de naissance = {JOUR_NAISS:02d}/{MOIS_NAISS:02d}/{ANNEE_NAISS}")
print(f"M (JJMMYYYY) = {M}")
print(f"n_rsa = {n_rsa} → M_mod = M % n_rsa = {M_mod}")
print(f"Chiffrement : C = M_mod^e_rsa mod n_rsa = {C}")
print(f"Déchiffrement : M_dec = C^d_rsa mod n_rsa = {M_dec}")
print("Vérification :", "OK  (M_dec == M_mod)" if M_dec == M_mod else " Problème — devrait retrouver M_mod")


===== Q4 : Chiffrement / Déchiffrement =====
Date de naissance = 15/07/2001
M (JJMMYYYY) = 15072001
n_rsa = 152415968983379 → M_mod = M % n_rsa = 15072001
Chiffrement : C = M_mod^e_rsa mod n_rsa = 1901943064234
Déchiffrement : M_dec = C^d_rsa mod n_rsa = 15072001
Vérification : OK  (M_dec == M_mod)


## PARTIE 3 : ECDSA

In [12]:
# =============================
# PARAMÈTRES (dérivés de la cellule "Paramètres utilisateur" de la partie 1)
# =============================
SECRET_DATE_S = f"{JOUR_NAISS:02d}/{MOIS_NAISS:02d}/{ANNEE_NAISS}"  # clé secrète (date)
MESSAGE_M     = SECRET_DATE_S                                      # message à signer

# Convertit "JJ/MM/AAAA" → JJMMAAAA (ex: "15/07/2001" → 15072001)
def date_to_int(datestr: str) -> int:
    jj, mm, yyyy = datestr.split("/")
    return int(jj)*1_000_000 + int(mm)*10_000 + int(yyyy)  # JJ*10^6 + MM*10^4 + YYYY

s_secret = date_to_int(SECRET_DATE_S)  # clé privée

# NB : is_probable_prime, nearest_prime, factorize (partie 1) et egcd, modinv (partie 2)
# sont réutilisés tels quels.


# =============================
# OUTILS : racines carrées mod p
# =============================
def legendre_symbol(a: int, p: int) -> int:
    """(a|p) ≡ a^((p-1)/2) mod p ∈ {0,1,p-1} → retourne 0,1,-1."""
    ls = pow(a, (p - 1) // 2, p)
    if ls == p - 1:
        return -1
    return ls

def tonelli_shanks(n: int, p: int) -> int:
    """Résout x^2 ≡ n (mod p) pour p impair premier. Retourne une racine ou lève ValueError."""
    assert p % 2 == 1 and is_probable_prime(p)
    if n % p == 0:
        return 0
    if legendre_symbol(n, p) != 1:
        raise ValueError("Non résoluble : n n'est pas un résidu quadratique mod p.")
    if p % 4 == 3:
        return pow(n, (p + 1) // 4, p)

    # Décomposer p-1 = q * 2^s avec q impair
    q = p - 1
    s = 0
    while q % 2 == 0:
        q //= 2
        s += 1
    # Trouver z non-résidu
    z = 2
    while legendre_symbol(z, p) != -1:
        z += 1
    m = s
    c = pow(z, q, p)
    t = pow(n, q, p)
    r = pow(n, (q + 1) // 2, p)
    while t != 1:
        i = 1
        t2i = pow(t, 2, p)
        while t2i != 1:
            t2i = pow(t2i, 2, p)
            i += 1
            if i == m:
                raise RuntimeError("Échec Tonelli-Shanks (improbable).")
        b = pow(c, 1 << (m - i - 1), p)
        r = (r * b) % p
        t = (t * b * b) % p
        c = (b * b) % p
        m = i
    return r


# =============================
# COURBE E : y^2 = x^3 + 1 (mod p)
# =============================
O = None  # point à l'infini

def is_on_curve(P, p):
    if P is None:
        return True
    x, y = P
    return (y * y - (x * x * x + 1)) % p == 0

def ec_neg(P, p):
    """-P = (x, -y)."""
    if P is None:
        return O
    x, y = P
    return (x, (-y) % p)

def ec_add(P, Q, p):
    if P is None: return Q
    if Q is None: return P
    x1, y1 = P; x2, y2 = Q
    if x1 == x2 and (y1 + y2) % p == 0:
        return O  # P + (-P) = O
    if P != Q:
        s = ((y2 - y1) * modinv((x2 - x1) % p, p)) % p
    else:
        # tangente (doublage) : s = (3x1^2 + a) / (2y1) ; ici a = 0
        s = ((3 * x1 * x1) * modinv((2 * y1) % p, p)) % p
    x3 = (s * s - x1 - x2) % p
    y3 = (s * (x1 - x3) - y1) % p
    return (x3, y3)

def ec_mul(k: int, P, p):
    """Double-and-add : k * P (k ≥ 0)."""
    R = O
    Q = P
    e = k
    while e > 0:
        if e & 1:
            R = ec_add(R, Q, p)
        Q = ec_add(Q, Q, p)
        e >>= 1
    return R

def find_point_high_order(p, max_tries=50, order_check_ceiling=3600):
    """
    Cherche un point P tel que pour tout 2 ≤ m ≤ order_check_ceiling, mP ≠ O.
    Cela garantit "ordre > 3600" (sans connaître l'ordre exact).
    On tente jusqu'à max_tries abscisses.
    """
    tried = 0
    x = 0
    while tried < max_tries and x < p:
        rhs = (pow(x, 3, p) + 1) % p
        if rhs != 0 and legendre_symbol(rhs, p) == 1:
            y = tonelli_shanks(rhs, p)
            for ycand in (y, (-y) % p):
                P = (x, ycand)
                if not is_on_curve(P, p):
                    continue
                # On calcule P, 2P, 3P, ... en ajoutant P à chaque pas (1 addition par m)
                ok = True
                R = P
                for m in range(2, order_check_ceiling + 1):
                    R = ec_add(R, P, p)
                    if R is O:
                        ok = False
                        break
                if ok:
                    return P
        x += 1
        tried += 1
    raise RuntimeError("Aucun point d'ordre > 3600 trouvé avec les limites choisies. Augmenter max_tries ou réduire le seuil.")

def point_order(P, p):
    """
    Ordre exact de P.
    1) Hasse : #E ∈ [p+1-2√p, p+1+2√p] et #E·P = O. On trouve un multiple m de l'ordre
       dans cet intervalle par baby-step giant-step (≈ 2·p^(1/4) opérations au lieu de p).
    2) On retire de m tous les facteurs premiers q tels que (m/q)·P = O → ordre exact.
    """
    if P is O:
        return 1
    w = 2 * isqrt(p) + 2                  # demi-largeur de l'intervalle de Hasse
    lo = p + 1 - w
    nb = isqrt(2 * w) + 1                 # nombre de pas de bébé

    # Pas de bébé : j·P pour 0 ≤ j < nb
    baby = {}
    R = O
    for j in range(nb):
        baby.setdefault(R, j)
        R = ec_add(R, P, p)
    # Pas de géant : on cherche lo + i·nb + j tel que (lo + i·nb)·P = -j·P
    giant = ec_neg(ec_mul(nb, P, p), p)
    T = ec_neg(ec_mul(lo, P, p), p)       # T = -(lo + i·nb)·P
    m = None
    for i in range(2 * w // nb + 2):
        if T in baby:
            m = lo + i * nb + baby[T]
            break
        T = ec_add(T, giant, p)
    if m is None:
        raise RuntimeError("Ordre introuvable dans l'intervalle de Hasse.")

    for q in factorize(m):
        while m % q == 0 and ec_mul(m // q, P, p) is O:
            m //= q
    return m


### QUESTION 1 : On considère la courbe y^2 = x^3+1 sur Z/pZ ou p est le nombre le plus proche de votre numéro d'étudiant. Trouver un point P d'ordre "élevé" (>3600 = 60^2)

In [13]:

p = nearest_prime(N_ETUDIANT, tie_break_to_higher=TIE_BREAK_TO_HIGHER)

print("===== Q1 : Paramètres de la courbe =====")
print(f"N = {N_ETUDIANT} → p (premier le plus proche) = {p}")
print("Courbe : E : y^2 = x^3 + 1 (mod p)")

P = find_point_high_order(p, max_tries=80, order_check_ceiling=3600)
n_ec = point_order(P, p)
print(f"Point P trouvé : P = {P}")
print(f"Ordre exact de P : n = {n_ec} (> 3600 ✔, premier : {is_probable_prime(n_ec)})")




===== Q1 : Paramètres de la courbe =====
N = 12345678 → p (premier le plus proche) = 12345701
Courbe : E : y^2 = x^3 + 1 (mod p)
Point P trouvé : P = (3, 8816852)
Ordre exact de P : n = 6172851 (> 3600 ✔, premier : False)


### QUESTION 2 : Soit s = date de naissance (JJ/MM/AAAA) la clé secrète. Calculer  Q=sP la clé publique

In [14]:

s = s_secret
Q = ec_mul(s, P, p)

print("\n===== Q2 : Clé secrète et clé publique =====")
print(f"s (privé) = {s}")
print(f"Q = sP = {Q}")
assert is_on_curve(Q, p), "Q n'est pas sur la courbe — erreur."




===== Q2 : Clé secrète et clé publique =====
s (privé) = 15072001
Q = sP = (9451938, 11041015)


### QUESTION 3 : Signer le message M = date de naissance par ECDSA avec les paramètres ci-dessus.

> **Remarque.** La courbe y² = x³ + 1 possède toujours des points de 2- et 3-torsion, donc l'ordre de P est en général composé (et souvent friable). L'ECDSA « standard » exige un sous-groupe d'ordre **premier** : ici on signe dans ⟨P⟩ en vérifiant à chaque fois que k et s sont inversibles modulo n. C'est suffisant pour l'exercice, mais un ordre friable rend le logarithme discret facile (Pohlig-Hellman) : à ne pas reproduire en vrai.


In [15]:
# ECDSA sur le sous-groupe <P> d'ordre n_ec.
# - h = SHA-256(M), tronqué à la taille en bits de n_ec (comme dans la norme)
# - signature : k aléatoire, R = kP, r = x_R mod n, s = k^(-1)(h + d·r) mod n
# - vérification : u1 = h·s^(-1), u2 = r·s^(-1), X = u1·P + u2·Q, valide si x_X ≡ r (mod n)
# Ici n_ec n'est en général PAS premier (voir remarque ci-dessus) : k et s doivent être
# inversibles mod n_ec, on retire donc un k tant que ce n'est pas le cas (nombre d'essais borné).

def hash_to_int(message: str, n: int) -> int:
    digest = int.from_bytes(hashlib.sha256(message.encode()).digest(), "big")
    return digest >> max(0, 256 - n.bit_length())

def ecdsa_sign(message, d, P, n, p, rng, max_tries=1000):
    h = hash_to_int(message, n)
    for _ in range(max_tries):
        k = rng.randrange(1, n)
        if gcd(k, n) != 1:
            continue
        R = ec_mul(k, P, p)
        r = R[0] % n
        if r == 0:
            continue
        sig_s = (modinv(k, n) * (h + d * r)) % n
        if sig_s == 0 or gcd(sig_s, n) != 1:
            continue
        return r, sig_s
    raise RuntimeError(
        "Signature impossible : h + d·r partage toujours un facteur avec n_ec. "
        "Changer de point P, de clé ou de message.")

def ecdsa_verify(message, signature, Q, P, n, p):
    r, sig_s = signature
    if not (1 <= r < n and 1 <= sig_s < n):
        return False
    h = hash_to_int(message, n)
    w = modinv(sig_s, n)
    X = ec_add(ec_mul(h * w % n, P, p), ec_mul(r * w % n, Q, p), p)
    return X is not O and X[0] % n == r

# Graine fixe pour des résultats reproductibles. En vrai, k doit être imprévisible
# (module secrets) ou dérivé de façon déterministe (RFC 6979) : réutiliser k révèle la clé privée.
rng = random.Random(2024)
signature = ecdsa_sign(MESSAGE_M, s, P, n_ec, p, rng)

print("\n===== Q3 : Signature ECDSA =====")
print(f"M = {MESSAGE_M!r}")
print(f"h = SHA-256(M) tronqué = {hash_to_int(MESSAGE_M, n_ec)}")
print(f"Signature (r, s) = {signature}")
print("Vérification (bon message)  :", "OK ✔" if ecdsa_verify(MESSAGE_M, signature, Q, P, n_ec, p) else "ÉCHEC ✘")
print("Vérification (message altéré):", "acceptée ✘" if ecdsa_verify(MESSAGE_M + "!", signature, Q, P, n_ec, p) else "rejetée ✔")


===== Q3 : Signature ECDSA =====
M = '15/07/2001'
h = SHA-256(M) tronqué = 2756834
Signature (r, s) = (6083121, 4952944)
Vérification (bon message)  : OK ✔
Vérification (message altéré): rejetée ✔
